# MODULE 04 · MINI PROJECT 01 — 종목 시세 및 재무정보 수집·정리 (2조)

## K-뷰티 수출 붐, 은행의 눈으로 본 3사
**핵심 질문:** 화장품 수출이 크게 늘고 있다. 이 붐을 만드는 3사는 은행 입장에서 믿을 만한 고객인가? 은행은 누구에게, 어떤 금융을 제공해야 하나?

분석 대상: 아모레퍼시픽(브랜드) · 코스맥스(ODM) · 에이피알(D2C 출발 글로벌 브랜드) — 밸류체인 위치는 **주 수익원 기준**

| 구분 | 출처 | 방법 | 원본 파일 |
|---|---|---|---|
| 일별 시세 | 네이버증권 차트 데이터(XML) | requests + **BeautifulSoup** | `data/raw/daily_price_raw.csv` |
| 투자지표(시총·PER·PBR) | 네이버증권 종목분석(FnGuide 제공) | requests + **BeautifulSoup** (막히면 네이버증권 JSON으로 보완) | `data/raw/price_snapshot.csv` |
| 재무(매출·영업이익·순이익·자본·부채) | 금융감독원 **OpenDART API** 주요계정 | requests (JSON) | `data/raw/financial_raw.csv` |
| 은행 여신 지표용 전체 재무제표(차입금·이자·현금흐름·매출채권·재고) | 금융감독원 **OpenDART API** 전체 재무제표 | requests (JSON) | `data/raw/financial_full_raw.csv` |
| 해외 매출 비중 | 각 사 실적 발표(수기) | 출처 URL 기록 | `data/raw/manual_overseas.csv` |

**실행 방법:** 메뉴 `런타임 → 모두 실행`
- DART 키: Colab 왼쪽 🔑(보안 비밀)에 이름 `DART_API_KEY`로 등록해 두면 자동으로 읽습니다. 없으면 실행 중 입력창이 뜨고, 빈칸으로 Enter를 누르면 저장된 원본 CSV를 사용합니다(캐시 모드).
- 어떤 수집이 실패해도 `data/raw`에 저장된 원본이 있으면 그걸로 끝까지 실행됩니다 → ZIP만 받은 사람도 재실행 가능.
- 마지막 STEP 5에서 Streamlit 앱(`app/app.py`)을 띄웁니다. 내 컴퓨터에서는 프로젝트 폴더에서 `streamlit run app/app.py`.

## 0. 준비 — 라이브러리 설치, 폴더, 공통 설정

In [ ]:
%pip install -q koreanize-matplotlib

In [ ]:
import os, re, io, time, zipfile, warnings
import xml.etree.ElementTree as ET
from datetime import datetime
from zoneinfo import ZoneInfo

# ── 작업 폴더: Colab이면 구글 드라이브, 내 컴퓨터면 프로젝트 폴더 ──
try:
    from google.colab import drive                      # Colab에서만 있는 모듈
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/MODULE04_미니프로젝트01_2조'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
except ImportError:                                      # 로컬 Jupyter에서 실행할 때
    if os.path.basename(os.getcwd()) == 'source':
        os.chdir('..')

for d in ['source', 'data/raw', 'data/processed', 'app', 'output', 'report', 'presentation']:
    os.makedirs(d, exist_ok=True)

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

KST = ZoneInfo('Asia/Seoul')                             # Colab 서버 시계는 한국 시간이 아님
COLLECTED_AT = datetime.now(KST).strftime('%Y-%m-%d %H:%M')
HEADERS = {'User-Agent': 'Mozilla/5.0'}                  # 브라우저처럼 보이게 (없으면 막힐 수 있음)
RAW, PROC, OUT = 'data/raw', 'data/processed', 'output'

def mask_key(x):
    '''오류 문구에 API 키가 섞여 나오지 않게 가림 (requests 오류에는 요청 주소가 통째로 들어감)'''
    text = re.sub(r'crtfc_key=[^&\s\'"]+', 'crtfc_key=***', str(x))
    key = globals().get('DART_KEY')
    return text.replace(key, '***') if key else text


STATUS = []                                              # 수집 성공·실패 기록 → data/raw/collection_status.csv
def log_status(source, code, ok, detail=''):
    STATUS.append({'출처': source, '종목코드': code, '결과': '성공' if ok else '실패',
                   '내용': mask_key(detail)[:150], '실행시각': COLLECTED_AT})
    pd.DataFrame(STATUS).to_csv(f'{RAW}/collection_status.csv', index=False, encoding='utf-8-sig')

print('작업 폴더 :', os.getcwd())
print('실행 시각 :', COLLECTED_AT, '(KST)')

## STEP 1. 분석 종목 선정
**왜 K-뷰티인가 (은행 관점 정성 비교)** — 수출 붐 업종 중 은행이 만나는 **서로 다른 유형의 기업고객**을 같은 기준으로 비교할 수 있는 업종을 골랐다.

| 기준 | 반도체·자동차 | K-푸드 | K-뷰티 |
|---|---|---|---|
| 수출 증가 | 큼 | 큼 | 큼 (2026 상반기 화장품 +27.3%) |
| 은행이 만나는 기업 유형 | 소수 대기업 중심 | 제조·브랜드 겸업 중심 | 브랜드 · ODM · 신흥 브랜드가 모두 상장 |
| 이번 분석과의 적합성 | 기업 규모 차이가 커 같은 기준 비교가 어려움 | 차순위 후보 (확장 대상) | 밸류체인 위치가 서로 다른 3사를 같은 기준으로 비교 가능 |

같은 K-뷰티 붐 안에서 **밸류체인 위치가 다른 3사**를 고름 — 브랜드 / 제조(ODM) / D2C로 출발한 글로벌 브랜드.
에이피알은 현재 아마존·오프라인 채널로도 팔기 때문에 '순수 D2C'가 아니라 'D2C 출발'로 표기함.
아모레퍼시픽은 지주사(아모레퍼시픽홀딩스, 002790)가 아닌 **사업회사(090430)** 로 확정해 지주사 할인을 제거함.
**밸류체인 위치는 각 사의 주 수익원(매출이 주로 어디서 나오는가) 기준**으로 구분함 — `주수익원` 컬럼에 함께 기록.

In [ ]:
STOCKS = pd.DataFrame([
    {'종목명': '아모레퍼시픽', '종목코드': '090430', '밸류체인': '브랜드',
     '주수익원': '자사 브랜드 화장품 판매 (국내·면세·해외)'},
    {'종목명': '코스맥스',     '종목코드': '192820', '밸류체인': 'ODM(제조)',
     '주수익원': '고객사 브랜드 제품의 개발·생산 (ODM 수주)'},
    {'종목명': '에이피알',     '종목코드': '278470', '밸류체인': '글로벌 브랜드(D2C 출발)',
     '주수익원': '자사 브랜드 화장품·뷰티 디바이스 판매 (해외 중심)'},
])
NAMES = STOCKS['종목명'].tolist()
STOCKS

## STEP 2. 종목 시세 수집 (네이버증권 · BeautifulSoup)

### 2-1. 트러블슈팅 기록 — 교안 코드가 안 된 이유
- 교안의 `finance.naver.com/item/main.naver` 는 **2026년 현재 새 사이트(`stock.naver.com`, Npay 증권)로 자동 이동**됨.
- 새 페이지는 화면을 자바스크립트로 그리기 때문에 HTML 안에 가격·PER 값이 없음 → `div.wrap_company` 같은 옛 선택자로는 '못 찾음'.
- 개발자도구(F12) → 네트워크 탭에서 실제 데이터 요청을 찾고, 경로별로 점검해 **HTML/XML로 값을 주는 경로**를 골라 BeautifulSoup을 그대로 사용함.

| 경로 | 결과 | 사용 |
|---|---|---|
| 새 페이지 HTML | 데이터 없음(JS 렌더링) | ✗ |
| 차트 데이터 `fchart.stock.naver.com` (XML) | 정상 | 일별 시세 |
| 종목분석 `navercomp.wisereport.co.kr` (HTML 표) | 정상 | 시총·PER·PBR |
| 네이버증권 JSON (`stock.naver.com/api/...`, `m.stock.naver.com/api/...`) | 정상 | 투자지표 백업 |

### 2-2. 일별 시세 (최근 약 1년) → `data/raw/daily_price_raw.csv`
원본은 **받은 그대로(문자열)** 저장하고, 숫자 변환은 STEP 4 전처리에서 함.

In [ ]:
def parse_chart_xml(content):
    '''네이버 차트 XML → [{날짜, 시가, 고가, 저가, 종가, 거래량}, ...]'''
    try:
        soup = BeautifulSoup(content, 'xml')            # XML은 xml 파서로 (lxml 필요)
    except Exception:
        soup = BeautifulSoup(content, 'html.parser')
    rows = []
    for item in soup.find_all('item'):
        date, open_, high, low, close, volume = item['data'].split('|')
        rows.append({'날짜': date, '시가': open_, '고가': high, '저가': low, '종가': close, '거래량': volume})
    return rows


def fetch_daily_prices(code, count=260):                # 1년 거래일 ≈ 250일
    url = f'https://fchart.stock.naver.com/sise.nhn?symbol={code}&timeframe=day&count={count}&requestType=0'
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    rows = parse_chart_xml(res.content)
    if not rows:
        raise ValueError(f'{code}: item 태그를 찾지 못함')
    return rows


daily_path = f'{RAW}/daily_price_raw.csv'
try:
    rows = []
    for _, s in STOCKS.iterrows():
        got = fetch_daily_prices(s['종목코드'])
        rows += [{'종목명': s['종목명'], '종목코드': s['종목코드'], **r} for r in got]
        log_status('네이버 차트(일별 시세)', s['종목코드'], True, f'{len(got)}일')
        time.sleep(1)                                    # 서버 부담 줄이기 (크롤링 예절)
    daily_raw = pd.DataFrame(rows)
    daily_raw['수집일시'] = COLLECTED_AT
    daily_raw.to_csv(daily_path, index=False, encoding='utf-8-sig')
    print('✅ 일별 시세 새로 수집')
except Exception as e:
    log_status('네이버 차트(일별 시세)', '-', False, e)
    print('⚠️ 수집 실패 → 저장된 원본 사용 |', e)
    if not os.path.exists(daily_path):
        raise RuntimeError('저장된 원본도 없음 — 인터넷 연결을 확인하고 다시 실행하세요') from e
    daily_raw = pd.read_csv(daily_path, dtype=str)

print(daily_raw.shape)
print(daily_raw.groupby('종목명')['날짜'].agg(['min', 'max', 'count']))
daily_raw.tail(3)

### 2-3. 투자지표 스냅샷 (시가총액·PER·추정PER·PBR·EPS·BPS) → `data/raw/price_snapshot.csv`
1순위: 네이버증권 **종목분석** 페이지(FnGuide 제공)를 BeautifulSoup으로 파싱
2순위: 비어 있는 값만 네이버증권 JSON으로 보완 (어느 경로에서 왔는지 `지표출처`에 기록)

In [ ]:
FIELDS = ['시가총액', 'PER', '추정PER', 'PBR', 'EPS', 'BPS']


def to_number(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    m = re.search(r'-?[\d,]*\.?\d+', str(text))
    return float(m.group().replace(',', '')) if m else np.nan


def money_to_eok(text):
    '''"16조 1,234억" / "161,234억원" / 13552612000000(원) → 억원'''
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    t = str(text).replace(',', '')
    jo, eok = re.search(r'([\d.]+)\s*조', t), re.search(r'([\d.]+)\s*억', t)
    if not jo and not eok:
        v = to_number(t)
        return v / 1e8 if pd.notna(v) and v > 1e10 else v   # 원 단위로 온 경우
    return (float(jo.group(1)) * 10000 if jo else 0) + (float(eok.group(1)) if eok else 0)


def parse_wisereport(html):
    '''종목분석 페이지 HTML → {항목: 원문값}'''
    soup = BeautifulSoup(html, 'html.parser')
    basic, ratios = {}, {}
    for tr in soup.select('table tr'):                   # 기본정보 표: th(항목) - td(값)
        th, td = tr.find('th'), tr.find('td')
        if th and td:
            basic.setdefault(th.get_text(' ', strip=True), td.get_text(' ', strip=True))
    for dl in soup.find_all('dl'):                       # PER·PBR 영역: <dl> 안에 '항목 값'
        m = re.match(r'(.+?)\s+(-?[\d,]+\.?\d*)', dl.get_text(' ', strip=True))
        if m:
            ratios.setdefault(m.group(1).strip(), m.group(2))

    def find(d, include, exclude=()):
        for k, v in d.items():
            if re.search(include, k) and not any(re.search(x, k) for x in exclude):
                return v
        return None

    return {
        '시가총액': find(basic, r'^시가총액'),
        'PER':     find(ratios, r'^PER', exclude=(r'12M', r'업종', r'추정')),
        '추정PER': find(ratios, r'(12M|추정)\s*PER'),
        'PBR':     find(ratios, r'^PBR'),
        'EPS':     find(ratios, r'^EPS'),
        'BPS':     find(ratios, r'^BPS'),
    }


def flatten_pairs(obj, out=None):
    '''JSON 어디에 있든 {"key": 이름, "value": 값} 과 {이름: 값} 을 모아 dict로'''
    out = {} if out is None else out
    if isinstance(obj, dict):
        label = obj.get('key') or obj.get('name') or obj.get('title')
        if label is not None and 'value' in obj and not isinstance(obj['value'], (dict, list)):
            out.setdefault(str(label), obj['value'])
            if obj.get('code'):
                out.setdefault(str(obj['code']), obj['value'])
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                flatten_pairs(v, out)
            elif k not in ('key', 'name', 'title', 'code', 'value'):
                out.setdefault(str(k), v)
    elif isinstance(obj, list):
        for v in obj:
            flatten_pairs(v, out)
    return out


JSON_LABELS = {
    '시가총액': ['시총', '시가총액', 'marketSum', 'marketValue'],
    'PER':     ['PER', 'per'],
    '추정PER': ['추정PER', 'cnsPer', 'estimatedPer', 'expectPer', 'forwardPer'],
    'PBR':     ['PBR', 'pbr'],
    'EPS':     ['EPS', 'eps'],
    'BPS':     ['BPS', 'bps'],
}


def parse_naver_json(data):
    flat = flatten_pairs(data)
    result = {}
    for field, labels in JSON_LABELS.items():
        for lb in labels:
            if flat.get(lb) not in (None, '', '-', 'N/A'):
                result[field] = flat[lb]
                break
    return result


def fetch_snapshot(code):
    got, source = {f: None for f in FIELDS}, {}
    # ① 종목분석(BeautifulSoup)
    try:
        url = f'https://navercomp.wisereport.co.kr/v2/company/c1010001.aspx?cmp_cd={code}'
        res = requests.get(url, headers=HEADERS, timeout=10)
        res.raise_for_status()
        parsed = parse_wisereport(res.content)
        for k, v in parsed.items():
            if v:
                got[k], source[k] = v, '종목분석(BS4)'
        log_status('종목분석(BS4)', code, any(parsed.values()), f'{sum(bool(v) for v in parsed.values())}개 항목')
    except Exception as e:
        log_status('종목분석(BS4)', code, False, e)
        print(f'   {code} 종목분석 실패: {e}')
    # ② 비어 있는 값만 네이버증권 JSON으로 보완
    for label, url in [
        ('네이버증권 JSON', f'https://stock.naver.com/api/domestic/detail/{code}/detail?codeType=KRX'),
        ('모바일 JSON',    f'https://m.stock.naver.com/api/stock/{code}/integration'),
    ]:
        if all(got.values()):
            break
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            res.raise_for_status()
            parsed = parse_naver_json(res.json())
            for k, v in parsed.items():
                if not got.get(k):
                    got[k], source[k] = str(v), label
                elif k in ('PER', 'PBR', '시가총액'):              # 두 출처 값 교차확인
                    conv = money_to_eok if k == '시가총액' else to_number
                    a, b = conv(got[k]), conv(v)
                    if pd.notna(a) and pd.notna(b) and b and abs(a / b - 1) > 0.03:
                        print(f'   ⚠️ {code} {k} 출처별 차이: {source[k]} {a:,.2f} vs {label} {b:,.2f}')
            log_status(label, code, bool(parsed), f'{len(parsed)}개 항목')
        except Exception as e:
            log_status(label, code, False, e)
            print(f'   {code} {label} 실패: {e}')
        time.sleep(0.5)
    return got, source


snap_path = f'{RAW}/price_snapshot.csv'
try:
    rows = []
    for _, s in STOCKS.iterrows():
        got, source = fetch_snapshot(s['종목코드'])
        row = {'종목명': s['종목명'], '종목코드': s['종목코드']}
        row.update({f'{k}_원문': v for k, v in got.items()})
        row['지표출처'] = ', '.join(f'{k}:{v}' for k, v in source.items())
        rows.append(row)
        time.sleep(1)
    snap_raw = pd.DataFrame(rows)
    snap_raw['수집일시'] = COLLECTED_AT
    if snap_raw[[f'{f}_원문' for f in ['시가총액', 'PER', 'PBR']]].isna().all().all():
        raise ValueError('시총·PER·PBR을 하나도 못 가져옴')
    snap_raw.to_csv(snap_path, index=False, encoding='utf-8-sig')
    print('✅ 투자지표 새로 수집')
except Exception as e:
    print('⚠️ 수집 실패 → 저장된 원본 사용 |', e)
    if not os.path.exists(snap_path):
        raise RuntimeError('저장된 원본도 없음 — 위에 찍힌 경로별 실패 메시지를 확인하세요') from e
    snap_raw = pd.read_csv(snap_path, dtype=str)

snap_raw

## STEP 3. 재무정보 수집 (금융감독원 OpenDART API)

- 공식 공시 원본이라 가장 신뢰도가 높음 (교안: "가능하면 공식 Open API 우선").
- API: 단일회사 주요계정 `fnlttSinglAcnt` — 매출액·영업이익·당기순이익·자본총계·부채총계·유동자산·유동부채, **연결(CFS)** 우선.
- 기간: 2024년 1분기 ~ 2026년 2분기 (보고서 코드: 11013=1분기, 11012=반기, 11014=3분기, 11011=사업보고서)
- **키는 코드에 쓰지 않음.** 🔑 보안 비밀 `DART_API_KEY` → 환경변수 → 입력창 순서로 찾고, 없으면 저장된 원본 사용.

In [ ]:
def get_dart_key():
    try:
        from google.colab import userdata
        key = userdata.get('DART_API_KEY')                # 괄호 안은 '보안 비밀 이름'
        if key:
            return key.strip()
    except Exception:
        pass
    if os.environ.get('DART_API_KEY'):
        return os.environ['DART_API_KEY'].strip()
    try:
        from getpass import getpass
        return getpass('DART API 키 붙여넣고 Enter (없으면 그냥 Enter → 저장된 원본 사용): ').strip()
    except Exception:
        return ''


def get_corp_codes(key, stock_codes):
    '''DART 고유번호(corp_code, 8자리) 찾기 — 종목코드(6자리)와 다름'''
    res = requests.get('https://opendart.fss.or.kr/api/corpCode.xml', params={'crtfc_key': key}, timeout=60)
    if not res.content.startswith(b'PK'):                # zip이 아니면 오류 메시지가 온 것
        raise RuntimeError('corpCode 오류: ' + res.text[:200])
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        root = ET.fromstring(z.read(z.namelist()[0]))
    mapping = {}
    for el in root.iter('list'):
        sc = (el.findtext('stock_code') or '').strip()
        if sc in stock_codes:
            mapping[sc] = el.findtext('corp_code')
    return mapping


def fetch_dart_accounts(key, corp_code, year, reprt_code):
    res = requests.get('https://opendart.fss.or.kr/api/fnlttSinglAcnt.json',
                       params={'crtfc_key': key, 'corp_code': corp_code,
                               'bsns_year': str(year), 'reprt_code': reprt_code}, timeout=20)
    data = res.json()
    if data.get('status') == '013':                      # 해당 보고서 없음
        return []
    if data.get('status') != '000':
        raise RuntimeError(f"DART 오류 {data.get('status')}: {data.get('message')}")
    return data['list']


REPORTS = [('11013', 1), ('11012', 2), ('11014', 3), ('11011', 4)]
PERIODS = [(y, rc, q) for y in (2024, 2025) for rc, q in REPORTS] + [(2026, rc, q) for rc, q in REPORTS[:2]]

fin_path = f'{RAW}/financial_raw.csv'
DART_KEY = get_dart_key()
try:
    if not DART_KEY:
        raise RuntimeError('DART 키 없음')
    corp = get_corp_codes(DART_KEY, set(STOCKS['종목코드']))
    print('DART 고유번호:', corp)
    rows = []
    for _, s in STOCKS.iterrows():
        n_before = len(rows)
        for year, rc, q in PERIODS:
            for item in fetch_dart_accounts(DART_KEY, corp[s['종목코드']], year, rc):
                rows.append({'종목명': s['종목명'], '종목코드': s['종목코드'], '연도': year, '분기': q, **item})
            time.sleep(0.3)
        log_status('OpenDART', s['종목코드'], len(rows) > n_before, f'{len(rows) - n_before}행')
    fin_raw = pd.DataFrame(rows)
    fin_raw['수집일시'] = COLLECTED_AT
    fin_raw.to_csv(fin_path, index=False, encoding='utf-8-sig')
    print('✅ DART 재무 새로 수집:', fin_raw.shape)
except Exception as e:
    log_status('OpenDART', '-', False, e)
    print('⚠️ DART 수집 안 함 → 저장된 원본 사용 |', mask_key(e))
    if not os.path.exists(fin_path):
        raise RuntimeError('첫 실행에는 DART 키가 필요합니다. 🔑에 DART_API_KEY를 등록하거나 입력창에 키를 넣고 다시 실행하세요') from e
    fin_raw = pd.read_csv(fin_path, dtype=str)

fin_raw[['종목명', '연도', '분기', 'fs_div', 'account_nm', 'thstrm_dt', 'thstrm_amount']].head(8)

### 3-2. 해외 매출 비중 (수기 입력) → `data/raw/manual_overseas.csv`
네이버증권·DART API에 구조화돼 있지 않아 **각 사 2026년 2분기 실적 발표**에서 직접 입력하고 출처를 남김.
파일이 이미 있으면 덮어쓰지 않음 → 담당자(조윤호)가 CSV를 직접 고치면 그 값이 반영됨.

In [ ]:
overseas_path = f'{RAW}/manual_overseas.csv'
if not os.path.exists(overseas_path):
    pd.DataFrame([
        {'종목명': '아모레퍼시픽', '종목코드': '090430', '기준분기': '2026Q2',
         '해외매출_억원': 5516, '전체매출_억원': 11759, '해외비중_%': 46.9,
         '산출방식': '해외 매출 ÷ 전체 매출 (사업회사 연결)',
         '출처URL': 'https://www.apgroup.com/int/ko/news/2026-07-30-1.html'},
        {'종목명': '코스맥스', '종목코드': '192820', '기준분기': '2026Q2',
         '해외매출_억원': 3050, '전체매출_억원': 8234, '해외비중_%': 37.0,
         '산출방식': '해외 법인(중국·미국·인니·태국) 매출 합 ÷ (국내+해외 법인 매출 합), 내부거래 제거 전 참고치',
         '출처URL': 'https://www.mt.co.kr/living/2026/08/11/2026081116023625031'},
        {'종목명': '에이피알', '종목코드': '278470', '기준분기': '2026Q2',
         '해외매출_억원': None, '전체매출_억원': 7675, '해외비중_%': 92.0,
         '산출방식': '회사 발표 해외 매출 비중',
         '출처URL': 'https://www.mt.co.kr/living/2026/08/05/2026080509392746396'},
    ]).to_csv(overseas_path, index=False, encoding='utf-8-sig')
    print('✅ 수기 데이터 파일 생성')
overseas_raw = pd.read_csv(overseas_path, dtype={'종목코드': str})
overseas_raw

### 3-3. 전체 재무제표 (은행 여신 지표용) → `data/raw/financial_full_raw.csv`
주요계정 API에는 **차입금·이자·현금흐름·매출채권·재고**가 없어서 전체 재무제표 API `fnlttSinglAcntAll`(연결)로 추가 수집.
- 필요한 보고서만 받음: 2025 사업보고서(연간) · 2025 반기 · 2026 반기
- 최근 4분기(TTM) 손익·현금흐름 = 2025 연간 + 2026 상반기 − 2025 상반기, 재무상태표는 2026 반기말 잔액

In [ ]:
FULL_REPORTS = [(2025, '11011'), (2025, '11012'), (2026, '11012')]
full_path = f'{RAW}/financial_full_raw.csv'


def fetch_dart_full(key, corp_code, year, reprt_code, fs_div='CFS'):
    res = requests.get('https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json',
                       params={'crtfc_key': key, 'corp_code': corp_code, 'bsns_year': str(year),
                               'reprt_code': reprt_code, 'fs_div': fs_div}, timeout=30)
    data = res.json()
    if data.get('status') == '013':                      # 해당 보고서 없음
        return []
    if data.get('status') != '000':
        raise RuntimeError(f"DART 오류 {data.get('status')}: {data.get('message')}")
    return data['list']


try:
    if not DART_KEY:
        raise RuntimeError('DART 키 없음')
    if 'corp' not in globals():
        corp = get_corp_codes(DART_KEY, set(STOCKS['종목코드']))
    rows = []
    for _, s in STOCKS.iterrows():
        n_before = len(rows)
        for year, rc in FULL_REPORTS:
            items = fetch_dart_full(DART_KEY, corp[s['종목코드']], year, rc, 'CFS')
            if not items:                                # 연결이 없으면 별도
                items = fetch_dart_full(DART_KEY, corp[s['종목코드']], year, rc, 'OFS')
            rows += [{'종목명': s['종목명'], '종목코드': s['종목코드'], '연도': year, **it, 'reprt_code': rc} for it in items]
            time.sleep(0.3)
        log_status('OpenDART 전체재무제표', s['종목코드'], len(rows) > n_before, f'{len(rows) - n_before}행')
    full_raw = pd.DataFrame(rows)
    full_raw['수집일시'] = COLLECTED_AT
    full_raw.to_csv(full_path, index=False, encoding='utf-8-sig')
    print('✅ 전체 재무제표 새로 수집:', full_raw.shape)
except Exception as e:
    log_status('OpenDART 전체재무제표', '-', False, e)
    print('⚠️ 전체 재무제표 수집 안 함 → 저장된 원본 사용 |', mask_key(e))
    full_raw = pd.read_csv(full_path, dtype=str) if os.path.exists(full_path) else pd.DataFrame()

if len(full_raw):
    print(full_raw.groupby(['종목명', '연도', 'reprt_code', 'sj_div']).size().unstack(fill_value=0))
else:
    print('전체 재무제표 없음 → 차입금·이자·현금흐름 지표는 계산하지 않고 건너뜀')

## STEP 4. 데이터 정리(전처리) 및 비교

### 4-1. 일별 시세 정제 → `data/processed/cleaned_price.csv`
- 문자열 → 숫자·날짜 변환, 결측·중복 확인
- **장중에 수집했다면 오늘 행은 종가가 아니므로 제외** (기준일 = 마지막 완성 거래일)
- 정규화지수(시작일=100), 1년 수익률, 연 변동성 계산

In [ ]:
price = daily_raw.copy()
price['종목코드'] = price['종목코드'].astype(str).str.zfill(6)
price['날짜'] = pd.to_datetime(price['날짜'].astype(str), format='%Y%m%d')
for c in ['시가', '고가', '저가', '종가', '거래량']:
    price[c] = pd.to_numeric(price[c].astype(str).str.replace(',', ''), errors='coerce')

print('결측값:', int(price[['시가', '고가', '저가', '종가', '거래량']].isna().sum().sum()),
      '| 중복 행:', int(price.duplicated(['종목코드', '날짜']).sum()))
price = price.dropna(subset=['종가']).drop_duplicates(['종목코드', '날짜'])

collected = pd.to_datetime(price['수집일시'].iloc[0])
if collected.hour * 60 + collected.minute < 15 * 60 + 40:   # 15:40 이전 = 장중
    today = price['날짜'].dt.date == collected.date()
    if today.any():
        print(f'⚠️ {collected:%H:%M} 장중 수집 → {collected:%Y-%m-%d} 행은 종가가 아니므로 제외')
        price = price[~today]

BASE_DATE = price['날짜'].max()
print('주가 기준일:', BASE_DATE.date())

common = price.groupby('날짜')['종목코드'].nunique()              # 3사 모두 거래한 날만 비교
price = price[price['날짜'].isin(common[common == len(STOCKS)].index)]
price = price.sort_values(['종목코드', '날짜'])
price['정규화지수'] = price.groupby('종목코드')['종가'].transform(lambda s: s / s.iloc[0] * 100)
price['일수익률'] = price.groupby('종목코드')['종가'].pct_change()

cleaned_price = price[['날짜', '종목코드', '종목명', '종가', '거래량', '정규화지수']].copy()
cleaned_price['날짜'] = cleaned_price['날짜'].dt.strftime('%Y-%m-%d')
cleaned_price['정규화지수'] = cleaned_price['정규화지수'].round(2)
cleaned_price.to_csv(f'{PROC}/cleaned_price.csv', index=False, encoding='utf-8-sig')

def price_summary(g):
    g = g.sort_values('날짜')
    last, prev, year = g.iloc[-1], g.iloc[-2], g.tail(250)
    return pd.Series({
        '현재가': last['종가'],
        '등락률': round((last['종가'] / prev['종가'] - 1) * 100, 2),
        '거래량': last['거래량'],
        '52주최고': year['고가'].max(),
        '52주최저': year['저가'].min(),
        '1년수익률': round((last['종가'] / g.iloc[0]['종가'] - 1) * 100, 1),
        '연변동성': round(g['일수익률'].std() * np.sqrt(252) * 100, 1),
    })

price_sum = price.groupby('종목명').apply(price_summary, include_groups=False).reindex(NAMES)
price_sum

### 4-2. 재무 데이터 정제 → `data/processed/quarterly_financials.csv`
DART 보고서는 **누적 기준**이 섞여 있음 (반기·3분기 보고서는 3개월치 + 누적, 사업보고서는 1년치).
- 분기 실적 = 이번 누적 − 직전 누적 → **4분기는 `연간 − 3분기 누적`으로 역산**
- 원 → 억원 변환, 연결(CFS) 우선·없으면 별도(OFS)
- 재무상태표 계정(자본·부채·유동자산·유동부채)은 분기말 잔액이라 그대로 사용 → **부채비율·유동비율**(은행 여신 관점의 재무 안정성)

In [ ]:
ACCOUNTS = {'매출액': r'^(매출액|수익\(매출액\)|영업수익)', '영업이익': r'^영업이익',
            '당기순이익': r'^당기순이익', '자본총계': r'^자본총계',
            '부채총계': r'^부채총계', '유동자산': r'^유동자산', '유동부채': r'^유동부채', '자산총계': r'^자산총계'}
BS_ACCOUNTS = {'자본총계', '부채총계', '유동자산', '유동부채', '자산총계'}     # 재무상태표 = 분기말 잔액


def to_eok(x):
    x = str(x).replace(',', '').strip()
    if x in ('', '-', 'nan', 'None', 'NaN'):
        return np.nan
    return float(x) / 1e8                                   # 원 → 억원


def months_of(dt_text):
    '''"2026.04.01 ~ 2026.06.30" → 3 (개월)'''
    d = re.findall(r'(\d{4})\.(\d{2})\.(\d{2})', str(dt_text))
    if len(d) >= 2:
        (y1, m1, _), (y2, m2, _) = d[0], d[1]
        return (int(y2) - int(y1)) * 12 + int(m2) - int(m1) + 1
    return None


def dart_to_quarterly(fin_raw):
    df = fin_raw.copy()
    df['종목코드'] = df['종목코드'].astype(str).str.zfill(6)
    df['연도'], df['분기'] = df['연도'].astype(int), df['분기'].astype(int)
    df['계정'] = None
    for acc, pat in ACCOUNTS.items():
        df.loc[df['account_nm'].astype(str).str.match(pat) & df['계정'].isna(), '계정'] = acc
    df = df.dropna(subset=['계정'])
    # 연결(CFS)이 있는 회사는 연결만, 없으면 별도(OFS)
    has_cfs = df.groupby('종목코드')['fs_div'].apply(lambda s: (s == 'CFS').any())
    df = df[df.apply(lambda r: r['fs_div'] == ('CFS' if has_cfs[r['종목코드']] else 'OFS'), axis=1)]
    df = df.drop_duplicates(['종목코드', '연도', '분기', '계정'])

    records = []
    for (name, code, acc, year), g in df.groupby(['종목명', '종목코드', '계정', '연도']):
        last_q, last_cum = 0, 0.0
        for _, r in g.sort_values('분기').iterrows():
            q = r['분기']
            amt = to_eok(r['thstrm_amount'])
            if acc in BS_ACCOUNTS:                            # 재무상태표: 분기말 잔액
                records.append((name, code, year, q, acc, amt))
                continue
            add = to_eok(r.get('thstrm_add_amount', np.nan))
            m = months_of(r.get('thstrm_dt', ''))
            if m is None:                                   # 기간 표기가 없으면 DART 관례로 판단
                m = 12 if q == 4 else 3
            three_m = amt if m == 3 else np.nan             # 3개월치 금액
            if q == 1:
                cum = amt
            elif not np.isnan(add) and q in (2, 3):
                cum = add                                   # 누적 금액
            elif m == 3 * q:
                cum = amt                                   # 연간(사업보고서) 등 누적
            elif not np.isnan(three_m) and last_q == q - 1:
                cum = last_cum + three_m
            else:
                cum = np.nan
            if not np.isnan(three_m):
                val = three_m
            elif last_q == q - 1 and not np.isnan(cum) and not np.isnan(last_cum):
                val = cum - last_cum                        # 예: 4분기 = 연간 - 3분기 누적
            else:
                val = np.nan
            records.append((name, code, year, q, acc, val))
            last_q, last_cum = q, cum

    long = pd.DataFrame(records, columns=['종목명', '종목코드', '연도', '분기', '계정', '값'])
    wide = long.pivot_table(index=['종목명', '종목코드', '연도', '분기'], columns='계정', values='값').reset_index()
    wide.columns.name = None
    wide['기간'] = wide['연도'].astype(str) + 'Q' + wide['분기'].astype(str)
    return wide.sort_values(['종목코드', '연도', '분기']).reset_index(drop=True)


qfin = dart_to_quarterly(fin_raw)
qfin['영업이익률'] = (qfin['영업이익'] / qfin['매출액'] * 100).round(1)
prev_year = qfin[['종목코드', '연도', '분기', '매출액']].rename(columns={'매출액': '전년동기매출액'})
prev_year['연도'] += 1
qfin = qfin.merge(prev_year, on=['종목코드', '연도', '분기'], how='left')
qfin['매출성장률_YoY'] = ((qfin['매출액'] / qfin['전년동기매출액'] - 1) * 100).round(1)
qfin = qfin.drop(columns='전년동기매출액')

# 분기별 3사 합계 대비 점유율 (3사 모두 값이 있는 분기만)
for col, share in [('매출액', '매출점유'), ('영업이익', '이익점유')]:
    total = qfin.groupby('기간')[col].transform('sum')
    cnt = qfin.groupby('기간')[col].transform('count')
    qfin[share] = np.where(cnt == len(STOCKS), (qfin[col] / total * 100).round(1), np.nan)

# 최근 4분기 합산 기준 점유율 — 한 분기의 일회성 손익에 덜 흔들리는 추세 지표
qfin = qfin.sort_values(['종목코드', '연도', '분기'])
for col in ['매출액', '영업이익']:
    qfin[f'{col}_4Q'] = qfin.groupby('종목코드')[col].transform(lambda s: s.rolling(4).sum())
for col, share in [('매출액_4Q', '매출점유_4Q'), ('영업이익_4Q', '이익점유_4Q')]:
    total = qfin.groupby('기간')[col].transform('sum')
    cnt = qfin.groupby('기간')[col].transform('count')
    qfin[share] = np.where(cnt == len(STOCKS), (qfin[col] / total * 100).round(1), np.nan)
qfin = qfin.drop(columns=['매출액_4Q', '영업이익_4Q'])

qfin['부채비율'] = (qfin['부채총계'] / qfin['자본총계'] * 100).round(1)   # 빚이 자본의 몇 %인가
qfin['유동비율'] = (qfin['유동자산'] / qfin['유동부채'] * 100).round(1)   # 1년 안에 갚을 빚 대비 1년 안에 현금이 될 자산
for c in ['매출액', '영업이익', '당기순이익', '자본총계', '부채총계', '유동자산', '유동부채', '자산총계']:
    qfin[c] = qfin[c].round(1)

print('결측값(계정별):'); print(qfin[['매출액', '영업이익', '당기순이익', '자본총계', '부채총계', '유동자산', '유동부채']].isna().sum())
qfin.to_csv(f'{PROC}/quarterly_financials.csv', index=False, encoding='utf-8-sig')
qfin.pivot_table(index='기간', columns='종목명', values='영업이익').reindex(columns=NAMES)

### 4-3. 수치 검증 — 공식 발표치와 대조
DART에서 계산한 2026년 2분기 실적이 각 사 실적 발표(IR·보도자료)와 같은지 자동 확인 (오차 1% 이내면 통과).

In [ ]:
IR_2026Q2 = {   # (매출액, 영업이익) 억원, 연결 — 각 사 2026년 2분기 실적 발표
    '아모레퍼시픽': (11759, 1173),
    '코스맥스':     (7949, 737),
    '에이피알':     (7675, 1906),
}
LATEST = '2026Q2'
check = qfin[qfin['기간'] == LATEST].set_index('종목명')
for name, (sales, op) in IR_2026Q2.items():
    for label, ir, val in [('매출액', sales, check.loc[name, '매출액'] if name in check.index else np.nan),
                           ('영업이익', op, check.loc[name, '영업이익'] if name in check.index else np.nan)]:
        diff = (val / ir - 1) * 100 if pd.notna(val) else np.nan
        mark = '✅' if pd.notna(diff) and abs(diff) <= 1 else '❌ 확인 필요'
        print(f'{mark} {name:7s} {label:5s} DART {val:>9,.0f} | 발표 {ir:>7,} | 차이 {diff:+.2f}%')

### 4-4. 은행 여신 지표 → `data/processed/bank_metrics.csv`
은행은 "주가가 오를까?"보다 **"빌려준 돈을 돌려받을 수 있나?"**를 먼저 본다. 전체 재무제표(3-3)로 상환 체력을 계산.

| 지표 | 계산 | 은행이 보는 이유 |
|---|---|---|
| 부채비율 | 부채총계 ÷ 자본총계 | 기본 재무 안정성 |
| 차입금의존도 | 총차입금 ÷ 자산총계 | 빚에 얼마나 기대고 있나 |
| 순차입금 | 총차입금 − 현금및현금성자산 | 음수면 순현금(빚보다 현금이 많음) |
| 이자보상배율 | 영업이익 ÷ 이자비용 (TTM) | 이익으로 이자를 감당하는가 |
| 순차입금/EBITDA | 순차입금 ÷ (영업이익 + 감가상각비) | 몇 년 벌어야 빚을 갚나 |
| 현금전환율 | 영업활동현금흐름 ÷ 영업이익 (TTM) | 장부상 이익이 실제 현금으로 들어오나 |
| 매출채권·재고 회전일수 | 잔액 ÷ TTM 매출 × 365 | 운전자금 수요, 부실 조기 신호 |

- 계정 이름이 회사마다 달라 **이름 규칙**으로 찾고, 어떤 계정을 썼는지 `사용계정`에 함께 저장
- 총차입금 = 단기·장기차입금 + 사채 + 유동성장기부채 (리스부채 제외). 차입금 계정 없이 `단기금융부채 · 장기금융부채`로만 공시한 회사는 그 금액으로 대신 (리스부채 등 포함 가능, 한계에 표기)
- 이자비용: 손익계산서 `이자비용` → 없으면 현금흐름표 조정 항목의 `이자비용`(손익계산서 이자비용과 같은 값) → 없으면 현금흐름표 `이자의 지급` → 그것도 없으면 `금융원가·금융비용`(이자 외 비용 포함, 한계에 표기)

In [ ]:
latest = qfin[qfin['기간'] == LATEST].set_index('종목명')
recent4 = ['2025Q3', '2025Q4', '2026Q1', '2026Q2']
prev4 = ['2024Q3', '2024Q4', '2025Q1', '2025Q2']
ttm = qfin[qfin['기간'].isin(recent4)].groupby('종목명')[['매출액', '영업이익', '당기순이익']].sum(min_count=4)
ttm_prev = qfin[qfin['기간'].isin(prev4)].groupby('종목명')[['매출액', '영업이익', '당기순이익']].sum(min_count=4)

IS_SJ, CF_SJ, BS_SJ = ['IS', 'CIS'], ['CF'], ['BS']


def full_rows(name, year, rc, sj):
    if full_raw.empty:
        return pd.DataFrame(columns=['account_nm', 'thstrm_amount', 'thstrm_add_amount'])
    f = full_raw[(full_raw['종목명'] == name) & (full_raw['연도'].astype(int) == year)
                 & (full_raw['reprt_code'].astype(str) == rc) & (full_raw['sj_div'].isin(sj))]
    return f.drop_duplicates(['account_nm'])


def pick(df, include, exclude=None, first=True):
    # 계정명(공백 제거) 규칙으로 행 찾기
    nm = df['account_nm'].astype(str).str.replace(r'\s', '', regex=True)
    m = nm.str.contains(include, regex=True)
    if exclude:
        m &= ~nm.str.contains(exclude, regex=True)
    hit = df[m]
    return hit.head(1) if first else hit


def amount(rows, cumulative=False):
    vals = []
    for _, r in rows.iterrows():
        v = to_eok(r.get('thstrm_add_amount')) if cumulative else np.nan
        if np.isnan(v):
            v = to_eok(r.get('thstrm_amount'))
        vals.append(v)
    vals = [v for v in vals if not np.isnan(v)]
    return float(np.sum(vals)) if vals else np.nan


def bs_bal(name, include, exclude=None, first=True):
    rows = pick(full_rows(name, 2026, '11012', BS_SJ), include, exclude, first)
    return amount(rows), ', '.join(rows['account_nm'].astype(str))


def ttm_flow(name, include, exclude, sj, first=True, absval=False):
    # 최근 4분기 = 2025 연간 + 2026 상반기 - 2025 상반기 (반기 손익은 누적 칸, 현금흐름은 원래 누적)
    total, used = 0.0, set()
    for year, rc, sign in [(2025, '11011', 1), (2026, '11012', 1), (2025, '11012', -1)]:
        rows = pick(full_rows(name, year, rc, sj), include, exclude, first)
        v = amount(rows, cumulative=(rc == '11012' and sj == IS_SJ))
        if np.isnan(v):
            return np.nan, ''
        total += sign * (abs(v) if absval else v)
        used |= set(rows['account_nm'].astype(str))
    return total, ', '.join(sorted(used))


bank_rows, bank = [], {}
for n in NAMES:
    debt, debt_acc = bs_bal(n, r'차입금|사채|유동성장기부채',
                            r'할인|조정|할증|파생|이자|리스|충당|전환권|교환권|상환권', first=False)
    if np.isnan(debt):                                   # 차입금 대신 '단기·장기금융부채'로만 공시한 경우
        debt, debt_acc = bs_bal(n, r'^(?:단기|장기|유동|비유동)금융부채$', None, first=False)
        if pd.notna(debt):
            debt_acc += ' (차입금 계정이 따로 없어 금융부채로 대신 — 리스부채 등 포함 가능)'
    if np.isnan(debt) and not full_raw.empty:
        debt, debt_acc = 0.0, '차입금 계정 없음 → 0'
    cash, cash_acc = bs_bal(n, r'^현금및현금성자산')
    assets = latest.loc[n, '자산총계'] if '자산총계' in latest else np.nan
    recv, recv_acc = bs_bal(n, r'^(?:단기)?매출채권', r'장기')
    inv, inv_acc = bs_bal(n, r'^재고자산')

    interest, int_acc = ttm_flow(n, r'^이자비용', None, IS_SJ)
    int_src = '손익계산서 이자비용'
    if np.isnan(interest) or interest <= 0:            # 현금흐름표 '조정' 칸의 이자비용 = 손익계산서 이자비용
        interest, int_acc = ttm_flow(n, r'^이자비용(?:에대한)?(?:조정)?$', None, CF_SJ, absval=True)
        int_src = '현금흐름표 조정 항목 이자비용'
    if np.isnan(interest) or interest <= 0:
        interest, int_acc = ttm_flow(n, r'이자(?:의)?지급', r'수취|수입|수익', CF_SJ, absval=True)
        int_src = '현금흐름표 이자 지급액'
    if np.isnan(interest) or interest <= 0:
        interest, int_acc = ttm_flow(n, r'^금융(?:원가|비용)', None, IS_SJ)
        int_src = '금융원가·금융비용'
    ocf, ocf_acc = ttm_flow(n, r'영업활동.*현금흐름', r'투자|재무', CF_SJ)
    da, da_acc = ttm_flow(n, r'상각비', r'대손|환입|손상|처분', CF_SJ, first=False)

    op, sales = ttm.loc[n, '영업이익'], ttm.loc[n, '매출액']
    net_debt = debt - cash if pd.notna(debt) and pd.notna(cash) else np.nan
    ebitda = op + da if pd.notna(da) else np.nan
    m = {
        '총차입금': debt, '현금성자산': cash, '순차입금': net_debt,
        '차입금의존도': debt / assets * 100 if pd.notna(debt) and pd.notna(assets) and assets else np.nan,
        '이자비용_TTM': interest,
        '이자보상배율': op / interest if pd.notna(interest) and interest > 0 else np.nan,
        'EBITDA_TTM': ebitda,
        '순차입금_EBITDA': net_debt / ebitda if pd.notna(net_debt) and pd.notna(ebitda) and ebitda > 0 else np.nan,
        '영업CF_TTM': ocf,
        '현금전환율': ocf / op * 100 if pd.notna(ocf) and op > 0 else np.nan,
        '매출채권회전일수': recv / sales * 365 if pd.notna(recv) else np.nan,
        '재고회전일수': inv / sales * 365 if pd.notna(inv) else np.nan,
        '영업이익률_TTM': op / sales * 100,
    }
    bank[n] = m
    src = {'총차입금': debt_acc, '현금성자산': cash_acc, '이자비용_TTM': f'{int_src}: {int_acc}' if int_acc else '',
           '영업CF_TTM': ocf_acc, 'EBITDA_TTM': f'영업이익 + {da_acc}' if da_acc else '',
           '매출채권회전일수': recv_acc, '재고회전일수': inv_acc}
    for k, v in m.items():
        bank_rows.append({'종목명': n, '지표': k, '값': round(v, 2) if pd.notna(v) else np.nan, '사용계정': src.get(k, '')})

bank = pd.DataFrame(bank).T.reindex(NAMES).astype(float).round(1)
bank_long = pd.DataFrame(bank_rows)
bank_long.to_csv(f'{PROC}/bank_metrics.csv', index=False, encoding='utf-8-sig')
used = bank_long[bank_long['사용계정'] != '']
if len(used):
    print('어떤 계정을 썼는지 (회사마다 이름이 달라 확인용):')
    print(used.pivot_table(index='지표', columns='종목명', values='사용계정', aggfunc='first').reindex(columns=NAMES))
bank

### 4-5. 종목별 비교 데이터 → `data/processed/comparison_data.csv`
| 컬럼 | 뜻 |
|---|---|
| 밸류체인 · 주수익원 | 밸류체인 위치는 주 수익원(매출이 주로 어디서 나오는가) 기준 |
| 매출액·영업이익·순이익 | 2026년 2분기(연결, 억원) |
| 매출점유·이익점유 | 3사 합계 대비 비중(%) — 분기 기준, `_TTM`은 최근 4분기 합계 기준 |
| 영업이익률 · 영업이익률_TTM | 영업이익 ÷ 매출액 (2026Q2 / 최근 4분기) |
| ROE | 최근 4분기 순이익 ÷ 2026Q2 자본총계 |
| 부채비율 · 유동비율 | 2026Q2말 부채총계 ÷ 자본총계, 유동자산 ÷ 유동부채 |
| 차입금의존도 · 순차입금 · 이자보상배율 · 순차입금_EBITDA · 현금전환율 · 회전일수 | 4-4 은행 여신 지표 |
| 순이익성장률 | 최근 4분기(2025Q3~2026Q2) 순이익 ÷ 직전 4분기 − 1 — 일회성 손익 영향이 커서 참고용 |
| PER·PBR·추정PER | 네이버증권 투자지표(KRX 기준). PER은 현재 EPS, 추정PER은 증권사 컨센서스 EPS 기준 |
| 예상EPS증가율 | 추정EPS ÷ 현재EPS − 1 = PER ÷ 추정PER − 1 (컨센서스 기준) |
| PER_계산 | 시가총액 ÷ 최근 4분기 순이익 — 비지배지분 포함 순이익이라 네이버 PER(지배주주 기준)과 차이 날 수 있음 |
| PEG | PER ÷ 예상EPS증가율(%) — 1보다 작을수록 성장 대비 낮게 평가. 이익 감소가 예상되면 계산하지 않음 |

In [ ]:
snap = snap_raw.copy()
snap['종목코드'] = snap['종목코드'].astype(str).str.zfill(6)
snap['시가총액'] = snap['시가총액_원문'].apply(money_to_eok)
for f in ['PER', '추정PER', 'PBR', 'EPS', 'BPS']:
    snap[f] = snap[f'{f}_원문'].apply(to_number)
snap = snap.set_index('종목명')
overseas = overseas_raw.set_index('종목명')

comp = STOCKS.set_index('종목명').copy()
comp = comp.join(price_sum[['현재가', '등락률', '거래량', '52주최고', '52주최저', '1년수익률', '연변동성']])
comp['매출액'] = latest['매출액']
comp['영업이익'] = latest['영업이익']
comp['순이익'] = latest['당기순이익']
comp['영업이익률'] = latest['영업이익률']
comp['매출성장률_YoY'] = latest['매출성장률_YoY']
comp['매출점유'] = latest['매출점유']
comp['이익점유'] = latest['이익점유']
comp['매출점유_TTM'] = (ttm['매출액'] / ttm['매출액'].sum() * 100).round(1)
comp['이익점유_TTM'] = (ttm['영업이익'] / ttm['영업이익'].sum() * 100).round(1)
comp['ROE'] = (ttm['당기순이익'] / latest['자본총계'] * 100).round(1)
comp['부채비율'] = latest['부채비율']
comp['유동비율'] = latest['유동비율']
for col in ['차입금의존도', '순차입금', '이자보상배율', '순차입금_EBITDA', '현금전환율',
            '매출채권회전일수', '재고회전일수', '영업이익률_TTM']:
    comp[col] = bank[col]
comp['순이익성장률'] = ((ttm['당기순이익'] / ttm_prev['당기순이익'] - 1) * 100).round(1)
comp['시가총액'] = snap['시가총액']
comp['PER'] = snap['PER']
comp['추정PER'] = snap['추정PER']
comp['PBR'] = snap['PBR']
comp['PER_계산'] = (comp['시가총액'] / ttm['당기순이익']).round(1)
comp['예상EPS증가율'] = ((comp['PER'] / comp['추정PER'] - 1) * 100).round(1)
comp['PEG'] = np.where(comp['예상EPS증가율'] > 0, (comp['PER'] / comp['예상EPS증가율']).round(2), np.nan)
comp['해외비중'] = overseas['해외비중_%']
comp['기준'] = f'재무 {LATEST} 연결(DART) / 주가 {BASE_DATE:%Y-%m-%d} 종가'
comp['수집일시'] = COLLECTED_AT

comparison = comp.reset_index()[[
    '종목명', '종목코드', '밸류체인', '주수익원', '현재가', '등락률', '거래량', '52주최고', '52주최저', '1년수익률', '연변동성',
    '매출액', '영업이익', '순이익', '영업이익률', '영업이익률_TTM', '매출성장률_YoY', 'ROE',
    '부채비율', '유동비율', '차입금의존도', '순차입금', '이자보상배율', '순차입금_EBITDA', '현금전환율',
    '매출채권회전일수', '재고회전일수',
    '시가총액', 'PER', '추정PER', 'PER_계산', 'PBR', '순이익성장률', '예상EPS증가율', 'PEG',
    '해외비중', '매출점유', '이익점유', '매출점유_TTM', '이익점유_TTM', '기준', '수집일시']]
comparison.to_csv(f'{PROC}/comparison_data.csv', index=False, encoding='utf-8-sig')
comparison.set_index('종목명').T

### 4-6. 여신 스코어카드 (과제용 단순 기준) → `data/processed/scorecard.csv`
**은행 내부 기준이 아닌 과제용 단순 기준**이며, 3사끼리 상대평가하지 않고 절대 구간으로 채점함.

| 영역 (지표) | 가중치 | 5점 | 4점 | 3점 | 2점 | 1점 |
|---|---|---|---|---|---|---|
| 상환 능력 (이자보상배율) | 30% | 10배 이상 또는 순현금 | 5~10배 | 3~5배 | 1~3배 | 1배 미만 |
| 재무 안정성 (부채비율) | 20% | 50% 미만 | 50~100% | 100~150% | 150~200% | 200% 이상 |
| 현금 창출 (현금전환율) | 20% | 100% 이상 | 80~100% | 60~80% | 40~60% | 40% 미만 |
| 수익성 (영업이익률, 최근 4분기) | 20% | 20% 이상 | 15~20% | 10~15% | 5~10% | 5% 미만 |
| 편중 리스크 (해외 매출 비중) | 10% | 40% 이하 | 40~60% | 60~80% | 80~90% | 90% 초과 |

종합 = 가중 평균 → **4.0점 이상 A · 3.0~4.0점 B · 3.0점 미만 C**. 값이 없는 지표는 나머지 가중치로 다시 나눔.
**투자자 순위**(1년 수익률·최근 4분기 이익 점유·PEG 평균 순위)와 **은행 순위**(스코어카드)를 함께 저장해 비교.

In [ ]:
SCORE_RULES = [   # (영역, 지표, 가중치, 구간 경계, 높을수록 좋은가, 경계값 비교)
    ('상환 능력', '이자보상배율', 0.30, [1, 3, 5, 10], True, '>='),
    ('재무 안정성', '부채비율', 0.20, [50, 100, 150, 200], False, '>='),
    ('현금 창출', '현금전환율', 0.20, [40, 60, 80, 100], True, '>='),
    ('수익성', '영업이익률_TTM', 0.20, [5, 10, 15, 20], True, '>='),
    ('편중 리스크', '해외비중', 0.10, [40, 60, 80, 90], False, '>'),
]


def band_score(v, cuts, higher_better, op):
    if pd.isna(v):
        return np.nan
    passed = sum((v >= c) if op == '>=' else (v > c) for c in cuts)
    return 1 + passed if higher_better else 5 - passed


d = comparison.set_index('종목명')
rows = []
for n in NAMES:
    r = {'종목명': n}
    got_w, got_s = 0.0, 0.0
    for area, col, w, cuts, hb, op in SCORE_RULES:
        v = d.loc[n, col]
        sc = band_score(v, cuts, hb, op)
        if col == '이자보상배율' and pd.notna(d.loc[n, '순차입금']) and d.loc[n, '순차입금'] < 0:
            sc = 5                                           # 순현금이면 상환 능력 최고점
        r[f'{area}_지표값'] = v
        r[f'{area}_점수'] = sc
        if pd.notna(sc):
            got_w += w
            got_s += w * sc
    r['종합점수'] = round(got_s / got_w, 2) if got_w else np.nan
    r['반영가중치'] = round(got_w, 2)
    r['등급'] = 'A' if r['종합점수'] >= 4 else 'B' if r['종합점수'] >= 3 else 'C'
    rows.append(r)
scorecard = pd.DataFrame(rows).set_index('종목명')

inv_rank = pd.concat([d['1년수익률'].rank(ascending=False), d['이익점유_TTM'].rank(ascending=False),
                      d['PEG'].rank(ascending=True, na_option='bottom')], axis=1).mean(axis=1)
scorecard['투자자순위'] = inv_rank.rank(method='min').astype(int)
scorecard['은행순위'] = scorecard['종합점수'].rank(ascending=False, method='min').astype(int)
scorecard.reset_index().to_csv(f'{PROC}/scorecard.csv', index=False, encoding='utf-8-sig')
scorecard[[c for c in scorecard.columns if c.endswith('점수')] + ['등급', '투자자순위', '은행순위', '반영가중치']]

## 시각화 (output/chart01~08.png)
색은 회사마다 고정: 에이피알=파랑, 아모레퍼시픽=주황, 코스맥스=청록 (색각이상 검증을 통과한 3색 조합). 모든 차트에 값 라벨을 직접 표시.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
try:
    import koreanize_matplotlib                            # 한글 폰트 (Colab)
except ImportError:
    import platform
    plt.rcParams['font.family'] = 'Malgun Gothic' if platform.system() == 'Windows' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {'에이피알': '#2a78d6', '아모레퍼시픽': '#eb6834', '코스맥스': '#1baf7a'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e2dc'


def style(ax, grid_axis='y'):
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=10)
    ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def save(fig, name):
    fig.savefig(f'{OUT}/{name}', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('저장:', f'{OUT}/{name}')

In [ ]:
# chart01 [메인] 매출 점유 vs 영업이익 점유 (2026년 2분기, 3사 합계 = 100%)
d = comparison.set_index('종목명')
fig, ax = plt.subplots(figsize=(10, 3.6))
for i, col in enumerate(['매출점유', '이익점유']):
    left = 0
    for name in NAMES:
        v = d.loc[name, col]
        ax.barh(i, v, left=left, color=COLORS[name], edgecolor='white', linewidth=2, height=0.55)
        if v >= 6:
            ax.text(left + v / 2, i, f'{name}\n{v:.1f}%', ha='center', va='center', fontsize=10, color=INK)
        left += v
ax.set_yticks([0, 1], ['매출 점유', '영업이익 점유'], fontsize=11, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
style(ax, grid_axis='x')
apr_s, apr_p = d.loc['에이피알', '매출점유'], d.loc['에이피알', '이익점유']
ax.set_title(f'매출 {apr_s:.0f}%인 에이피알이 3사 영업이익의 {apr_p:.0f}%를 가져갔다',
             loc='left', fontsize=14, color=INK, pad=24, fontweight='bold')
ax.text(0, 1.04, f'{LATEST} 연결 기준 · 3사 합계 = 100% · 출처: OpenDART', transform=ax.transAxes, fontsize=9, color=INK2)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLORS[n]) for n in NAMES], labels=NAMES,
          loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False, fontsize=9)
save(fig, 'chart01_profit_share.png')

In [ ]:
# chart02 영업이익 점유 추이 (최근 4분기 합산) — '한 분기만 고른 것 아니냐'는 반박에 대한 답
share = qfin.pivot_table(index='기간', columns='종목명', values='이익점유_4Q').dropna().reindex(columns=NAMES)
fig, ax = plt.subplots(figsize=(10, 4.2))
x = range(len(share))
for name in NAMES:
    ax.plot(x, share[name], color=COLORS[name], linewidth=2, marker='o', markersize=6,
            markeredgecolor='white', markeredgewidth=1.5, label=name)
    ax.text(len(share) - 1 + 0.15, share[name].iloc[-1], f'{name} {share[name].iloc[-1]:.1f}%',
            va='center', fontsize=10, color=INK)
ax.set_xticks(list(x), share.index, fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlim(-0.3, len(share) + 1.6)
style(ax)
ax.set_title('3사 영업이익 점유율 추이 (최근 4분기 합산, 3사 합계 = 100%)', loc='left', fontsize=14, color=INK, fontweight='bold', pad=20)
ax.text(0, 1.02, '각 시점마다 직전 4개 분기를 합쳐 계산 — 한 분기의 일회성 손익에 덜 흔들림', transform=ax.transAxes,
        fontsize=9, color=INK2)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9, ncol=3)
save(fig, 'chart02_profit_share_trend.png')

In [ ]:
# chart03 성장성 × 수익성 (점 크기 = 시가총액)
fig, ax = plt.subplots(figsize=(8, 5.2))
cap = d['시가총액'].fillna(d['시가총액'].median() if d['시가총액'].notna().any() else 1)
sizes = cap / cap.max() * 2400 + 200
pts = d[['매출성장률_YoY', '영업이익률']].astype(float)
span = (pts.max() - pts.min()).replace(0, 1)
for name in NAMES:
    xg, ym = pts.loc[name, '매출성장률_YoY'], pts.loc[name, '영업이익률']
    ax.scatter(xg, ym, s=sizes[name], color=COLORS[name], alpha=0.85, edgecolor='white', linewidth=2, zorder=3)
    cap_txt = f"{d.loc[name, '시가총액'] / 10000:.1f}조" if pd.notna(d.loc[name, '시가총액']) else '시총 없음'
    # 가장 가까운 다른 점의 반대쪽(위/아래)에 라벨 → 라벨끼리 겹치지 않게
    others = ((pts.drop(name) - pts.loc[name]) / span).pow(2).sum(axis=1)
    nb = pts.loc[others.idxmin()]
    up = nb['영업이익률'] < ym or (nb['영업이익률'] == ym and nb['매출성장률_YoY'] > xg)
    radius = np.sqrt(sizes[name]) / 2
    ax.annotate(f'{name}\n성장 {xg:.0f}% · 이익률 {ym:.1f}% · 시총 {cap_txt}', (xg, ym),
                textcoords='offset points', xytext=(0, (radius + 4) * (1 if up else -1)),
                ha='center', va='bottom' if up else 'top', fontsize=9, color=INK, zorder=4)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('매출 성장률 (전년 동기 대비)', color=INK2)
ax.set_ylabel('영업이익률', color=INK2)
ax.margins(x=0.25, y=0.45)
style(ax, grid_axis='both')
ax.set_title(f'성장성 × 수익성 ({LATEST}, 점 크기 = 시가총액)', loc='left', fontsize=14, color=INK, fontweight='bold')
save(fig, 'chart03_growth_margin.png')

In [ ]:
# chart04 PER vs PEG — 단위가 달라 한 축에 겹치지 않고 두 칸으로 나눔
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, col, title in [(axes[0], 'PER', 'PER (배)'), (axes[1], 'PEG', 'PEG = PER ÷ 예상 EPS 증가율(%)')]:
    vals = d[col].reindex(NAMES)
    ax.bar(NAMES, vals.fillna(0), color=[COLORS[n] for n in NAMES], edgecolor='white', linewidth=2, width=0.6)
    for i, v in enumerate(vals):
        ax.text(i, (v if pd.notna(v) else 0), f'{v:.2f}' if pd.notna(v) else '해당없음\n(이익 감소 예상)',
                ha='center', va='bottom', fontsize=10, color=INK)
    if col == 'PEG':
        ax.axhline(1, color=INK2, linewidth=1, linestyle='--')
        ax.text(2.45, 1, 'PEG 1', va='bottom', ha='right', fontsize=8, color=INK2)
    ax.set_title(title, loc='left', fontsize=12, color=INK)
    ax.margins(y=0.2)
    style(ax)
fig.suptitle('시장은 이익 차이를 가격에 반영했나', x=0.02, ha='left', fontsize=14, color=INK, fontweight='bold')
fig.tight_layout()
save(fig, 'chart04_per_peg.png')

In [ ]:
# chart05 1년 주가 흐름 (시작일 = 100)
idx = cleaned_price.pivot_table(index='날짜', columns='종목명', values='정규화지수').reindex(columns=NAMES)
idx.index = pd.to_datetime(idx.index)
fig, ax = plt.subplots(figsize=(10, 4.4))
for name in NAMES:
    s = idx[name].dropna()
    ax.plot(s.index, s, color=COLORS[name], linewidth=2, label=name)
    ax.text(s.index[-1], s.iloc[-1], f'  {name} {s.iloc[-1]:.0f}', va='center', fontsize=10, color=INK)
ax.axhline(100, color=INK2, linewidth=1, linestyle='--')
style(ax)
ax.set_title(f'1년 주가 흐름 (시작일 = 100, 기준일 {BASE_DATE:%Y-%m-%d})', loc='left', fontsize=14, color=INK, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), frameon=False, fontsize=9, ncol=3)
ax.set_xlim(idx.index.min(), idx.index.max() + pd.Timedelta(days=45))
save(fig, 'chart05_price_index.png')

In [ ]:
# chart06 은행 여신 지표 — 상환 체력 한눈에
panels = [('부채비율', '부채비율 (%)', 200, '200% (스코어 1점 경계)'),
          ('이자보상배율', '이자보상배율 (배)', 3, '3배'),
          ('현금전환율', '현금전환율 (%, 영업CF ÷ 영업이익)', 100, '100%'),
          ('차입금의존도', '차입금의존도 (%)', None, None)]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (col, title, ref, ref_label) in zip(axes.flat, panels):
    vals = d[col].reindex(NAMES)
    shown = vals.clip(upper=vals.dropna().median() * 4) if col == '이자보상배율' and vals.notna().sum() >= 2 else vals
    ax.bar(NAMES, shown.fillna(0), color=[COLORS[n] for n in NAMES], edgecolor='white', linewidth=2, width=0.6)
    for i, n in enumerate(NAMES):
        v = vals[n]
        net_cash = col == '이자보상배율' and pd.notna(d.loc[n, '순차입금']) and d.loc[n, '순차입금'] < 0
        label = '없음' if pd.isna(v) else f'{v:,.1f}' + (' (순현금)' if net_cash else '')
        ax.text(i, shown.fillna(0)[n], label, ha='center', va='bottom', fontsize=10, color=INK)
    if ref is not None:
        ax.axhline(ref, color=INK2, linewidth=1, linestyle='--')
        ax.text(2.45, ref, ref_label, va='bottom', ha='right', fontsize=8, color=INK2)
    ax.set_title(title, loc='left', fontsize=12, color=INK)
    ax.margins(y=0.2)
    style(ax)
fig.suptitle(f'은행의 눈: 상환 체력 비교 ({LATEST}말 잔액 · 최근 4분기 손익·현금흐름)', x=0.02, ha='left',
             fontsize=14, color=INK, fontweight='bold')
fig.tight_layout()
save(fig, 'chart06_bank_metrics.png')

In [ ]:
# chart07 여신 스코어카드 — 영역별 가중 점수를 쌓은 막대 (최대 5점)
areas = [a for a, *_ in SCORE_RULES]
weights = {a: w for a, _, w, *_ in SCORE_RULES}
shades = ['#1f4e79', '#2e75b6', '#6fa8dc', '#a9c9ea', '#d5e3f3']
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, n in enumerate(NAMES):
    left = 0
    total_w = scorecard.loc[n, '반영가중치'] or 1
    for a, c in zip(areas, shades):
        sc = scorecard.loc[n, f'{a}_점수']
        if pd.isna(sc):
            continue
        part = weights[a] * sc / total_w
        ax.barh(i, part, left=left, color=c, edgecolor='white', linewidth=1.5, height=0.55,
                label=f'{a} ({weights[a]:.0%})' if i == 0 else None)
        ax.text(left + part / 2, i, f'{sc:.0f}', ha='center', va='center', fontsize=9,
                color='white' if c in shades[:2] else INK)
        left += part
    ax.text(left + 0.06, i, f"{scorecard.loc[n, '종합점수']:.2f}점 · {scorecard.loc[n, '등급']}등급",
            va='center', fontsize=11, color=INK, fontweight='bold')
ax.set_yticks(range(len(NAMES)), NAMES, fontsize=11, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, 5.9)
for x, g in [(3, 'C | B'), (4, 'B | A')]:
    ax.axvline(x, color=INK2, linewidth=1, linestyle='--')
    ax.text(x, -0.55, g, ha='center', fontsize=8, color=INK2)
style(ax, grid_axis='x')
ax.set_title('여신 스코어카드 (과제용 단순 기준, 막대 안 숫자 = 영역별 1~5점)', loc='left', fontsize=14,
             color=INK, fontweight='bold', pad=22)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=5, frameon=False, fontsize=8)
save(fig, 'chart07_scorecard.png')

In [ ]:
# chart08 투자자 순위 vs 은행 순위 — 같은 회사를 두 눈으로 보면 순위가 바뀌는가
fig, ax = plt.subplots(figsize=(7, 4.6))
for n in NAMES:
    a, b = scorecard.loc[n, '투자자순위'], scorecard.loc[n, '은행순위']
    ax.plot([0, 1], [a, b], color=COLORS[n], linewidth=3, marker='o', markersize=10)
    ax.text(-0.04, a, f'{n}  {a}위', ha='right', va='center', fontsize=11, color=INK)
    ax.text(1.04, b, f'{b}위  {n} ({scorecard.loc[n, "등급"]})', ha='left', va='center', fontsize=11, color=INK)
ax.set_xlim(-0.75, 1.75)
ax.set_ylim(len(NAMES) + 0.5, 0.5)
ax.set_xticks([0, 1], ['투자자 순위\n(1년 수익률·이익 점유·PEG)', '은행 순위\n(여신 스코어카드)'], fontsize=10, color=INK)
ax.set_yticks([])
for side in ['top', 'right', 'left', 'bottom']:
    ax.spines[side].set_visible(False)
ax.set_title('투자자와 은행은 같은 회사를 다르게 본다', loc='left', fontsize=14, color=INK, fontweight='bold')
save(fig, 'chart08_rank_compare.png')

## 분석 결과 요약 (자동 생성 — 보고서·발표 문장 초안)
숫자는 위 데이터에서 바로 계산되므로, 데이터를 다시 수집하면 문장도 같이 바뀜.

In [ ]:
def f1(v, unit=''):
    return f'{v:,.1f}{unit}' if pd.notna(v) else '없음'

apr = d.loc['에이피알']
lines = [
    f"[Q1 이익 배분] {LATEST} 3사 합계에서 에이피알은 매출 {f1(apr['매출점유'], '%')}, 영업이익 {f1(apr['이익점유'], '%')} "
    f"→ 이익 점유가 매출 점유의 {apr['이익점유'] / apr['매출점유']:.2f}배.",
    f"          최근 4분기 합계 기준으로도 매출 {f1(apr['매출점유_TTM'], '%')} / 이익 {f1(apr['이익점유_TTM'], '%')}.",
]
for name in NAMES:
    r = d.loc[name]
    lines.append(f"   - {name}({r['밸류체인']}): 영업이익률 {f1(r['영업이익률'], '%')}, 매출 성장률 {f1(r['매출성장률_YoY'], '%')}, "
                 f"매출 점유 {f1(r['매출점유'], '%')} → 이익 점유 {f1(r['이익점유'], '%')}")
lines.append('[Q2 시장 평가] PER / PEG')
for name in NAMES:
    r = d.loc[name]
    lines.append(f"   - {name}: PER {f1(r['PER'], '배')} → 추정PER {f1(r['추정PER'], '배')}, 예상 EPS 증가율 {f1(r['예상EPS증가율'], '%')}, "
                 f"PEG {f1(r['PEG'])} (참고: 최근 4분기 순이익 증가율 {f1(r['순이익성장률'], '%')})")
lines.append('[Q3 지속성] 해외 매출 비중')
for name in NAMES:
    lines.append(f"   - {name}: {f1(d.loc[name, '해외비중'], '%')}")
lines.append('[재무 안정성] 부채비율 / 유동비율 (은행 여신 관점)')
for name in NAMES:
    lines.append(f"   - {name}: 부채비율 {f1(d.loc[name, '부채비율'], '%')}, 유동비율 {f1(d.loc[name, '유동비율'], '%')}")
lines.append('[상환 체력] 이자보상배율 / 현금전환율 / 차입금의존도 / 순차입금')
for name in NAMES:
    lines.append(f"   - {name}: 이자보상배율 {f1(d.loc[name, '이자보상배율'], '배')}, 현금전환율 {f1(d.loc[name, '현금전환율'], '%')}, "
                 f"차입금의존도 {f1(d.loc[name, '차입금의존도'], '%')}, 순차입금 {f1(d.loc[name, '순차입금'], '억원')}")
lines.append('[여신 스코어카드] 과제용 단순 기준')
for name in NAMES:
    lines.append(f"   - {name}: {scorecard.loc[name, '종합점수']:.2f}점 {scorecard.loc[name, '등급']}등급 "
                 f"(투자자 순위 {scorecard.loc[name, '투자자순위']}위 → 은행 순위 {scorecard.loc[name, '은행순위']}위)")
lines.append(f"(기준: {d['기준'].iloc[0]})")
share4 = qfin.pivot_table(index='기간', columns='종목명', values='이익점유_4Q').dropna()
if len(share4):
    lines.insert(2, f"          최근 4분기 합산 이익 점유 추이: 에이피알 {share4['에이피알'].iloc[0]:.1f}%({share4.index[0]}) "
                    f"→ {share4['에이피알'].iloc[-1]:.1f}%({share4.index[-1]})")
lines += ['※ 점유율은 분석 대상 3사 합계 안에서의 구성비이며 K-뷰티 산업 전체 점유율이 아님',
          '※ 순이익은 일회성 손익 영향이 커서 PEG는 증권사 컨센서스(추정 EPS) 기준 증가율로 계산',
          '※ 에이피알의 높은 이익률을 유통 구조(D2C) 하나로 단정할 수 없음 — 가설 수준',
          '※ 수업 과제용 분석이며 투자 권유 목적이 아님']
summary = '\n'.join(lines)
print(summary)
with open(f'{OUT}/analysis_summary.txt', 'w', encoding='utf-8') as fp:
    fp.write(summary)

## 금융·비즈니스 인사이트 (자동 생성 → `output/insights.md`)
숫자는 위 데이터에서 계산되므로 다시 수집하면 문장도 함께 갱신됨. 흐름: 성장의 질 → 상환 체력 → 신용 리스크 → 시장 평가 vs 은행 판단 → 은행의 판단. **은행(여신·기업금융) 관점** 중심 — 투자자는 "주가가 오를까?", 은행은 "빌려준 돈을 돌려받을 수 있나? 어떤 거래를 할 수 있나?"를 묻는다. Streamlit '인사이트' 탭과 보고서·발표의 결론에 그대로 사용.

In [ ]:
s4 = qfin.pivot_table(index='기간', columns='종목명', values='이익점유_4Q').dropna()
r4 = qfin.pivot_table(index='기간', columns='종목명', values='매출점유_4Q').dropna()
first, last = s4.index[0], s4.index[-1]
winner = s4.iloc[-1].idxmax()                            # 최근 4분기 합산 이익 점유 1위
sc = scorecard


def pct(v, signed=False):
    return '없음' if pd.isna(v) else (f'{v:+.1f}%' if signed else f'{v:.1f}%')


def times(n):
    v, nd = d.loc[n, '이자보상배율'], d.loc[n, '순차입금']
    if pd.notna(nd) and nd < 0:
        return '순현금(빚보다 현금이 많음)'
    return '없음' if pd.isna(v) else f'{v:.1f}배'


def move(n):
    return f"{s4[n].iloc[0]:.1f}% → {s4[n].iloc[-1]:.1f}%"


def josa(word, pair):
    # 받침 있으면 앞 글자(은/을/이), 없으면 뒤 글자(는/를/가)
    ch = word[-1]
    has_final = '가' <= ch <= '힣' and (ord(ch) - 0xAC00) % 28 != 0
    return word + (pair[0] if has_final else pair[1])


insights = []   # (제목, 숫자, 의미, 활용)

# 1. 성장의 질
insights.append((
    '성장의 질: 이익은 해외 판매 채널을 가진 브랜드로 이동',
    f"최근 4분기 합산 영업이익 점유({first} → {last}): " + ', '.join(f'{n} {move(n)}' for n in NAMES)
    + '. 최근 4분기 영업이익률: ' + ', '.join(f"{n} {pct(d.loc[n, '영업이익률_TTM'])}" for n in NAMES) + '.',
    f'같은 수출 붐 안에서도 이익은 {winner} 쪽으로 쏠린다. 이익률이 높을수록 이자를 감당할 여력이 크고, '
    '매출이 늘어도 이익 몫이 줄어드는 회사는 차입을 늘릴 때 부담이 커질 수 있다.',
    '여신 포트폴리오: 대형사 이익 점유를 분기마다 추적해 업종·협력사 여신 점검의 조기 경보 지표로 쓴다.'))

# 2. 상환 체력
weakest = d['부채비율'].idxmax()
cash_rank = d['현금전환율'].dropna().sort_values()
ccr_txt = (f"현금전환율이 가장 낮은 곳은 {cash_rank.index[0]}({pct(cash_rank.iloc[0])})" if len(cash_rank)
           else '현금전환율은 전체 재무제표가 없어 계산하지 못함')
fastest = d['매출성장률_YoY'].idxmax()
safest = scorecard['종합점수'].idxmax()
insights.append((
    ('상환 체력: 성장이 가장 빠른 회사가 곧 가장 안전한 차주는 아니다' if fastest != safest
     else f'상환 체력: 성장이 가장 빠른 {josa(fastest, "이가")} 상환 체력도 가장 좋다'),
    '; '.join(f"{n} 부채비율 {pct(d.loc[n, '부채비율'])} · 이자보상배율 {times(n)} · 현금전환율 {pct(d.loc[n, '현금전환율'])}"
              for n in NAMES) + '.',
    f"빚 부담이 가장 큰 곳은 {weakest}(부채비율 {pct(d.loc[weakest, '부채비율'])}, 유동비율 {pct(d.loc[weakest, '유동비율'])}). "
    f'{ccr_txt}. 장부상 이익이 현금으로 얼마나 들어오는지가 상환 재원의 질을 가른다.',
    '여신 심사: 이익 규모보다 이자보상배율·현금전환율을 먼저 보고, 약한 지표는 약정 조건(코버넌트)으로 관리한다.'))

# 3. 신용 리스크
top_ovs, top_vol = d['해외비중'].idxmax(), d['연변동성'].idxmax()
neg = [n for n in NAMES if pd.notna(d.loc[n, '예상EPS증가율']) and d.loc[n, '예상EPS증가율'] < 0]
insights.append((
    '신용 리스크: 무엇이 이 체력을 흔드나',
    f"해외 매출 비중 {top_ovs} {pct(d.loc[top_ovs, '해외비중'])}; 예상 EPS 감소 "
    + (', '.join(f"{n} {pct(d.loc[n, '예상EPS증가율'], True)}" for n in neg) if neg else '없음')
    + f"; 1년 주가 변동성 " + ', '.join(f"{n} {pct(d.loc[n, '연변동성'])}" for n in NAMES) + '.',
    f"{josa(top_ovs, '은는')} 환율·관세·해외 판매 채널 변화에 가장 민감하고, "
    + (f"{josa(', '.join(neg), '은는')} 이익 추정이 내려가는 중이라 상환 재원 둔화가 관전 포인트다." if neg
       else '이익 추정 하향은 아직 보이지 않는다.'),
    '리스크 관리: 지역·채널 편중에 따른 한도 관리, 이익 추정 하향을 조기 경보 신호로 활용한다.'))

# 4. 시장 평가 vs 은행 판단
per_rank = ' > '.join(f"{n} {d.loc[n, 'PER']:.1f}배" for n in d['PER'].dropna().sort_values(ascending=False).index)
flip = [n for n in NAMES if sc.loc[n, '투자자순위'] != sc.loc[n, '은행순위']]
inv_top = sc['투자자순위'].idxmin()
bank_top = sc['은행순위'].idxmin()
insights.append((
    '시장 평가 vs 은행 판단: 순위가 ' + ('바뀐다' if flip else '같다'),
    f'PER: {per_rank}. PEG: ' + ', '.join(f"{n} {d.loc[n, 'PEG']:.2f}" if pd.notna(d.loc[n, 'PEG']) else f'{n} 해당없음'
                                         for n in NAMES)
    + '. 순위(투자자 → 은행): ' + ', '.join(f"{n} {sc.loc[n, '투자자순위']}위 → {sc.loc[n, '은행순위']}위" for n in NAMES) + '.',
    (f'투자자 관점 1위는 {inv_top}, 은행 스코어카드 1위는 {bank_top}. 시장 평가는 미래에 대한 기대를, '
     '은행 판단은 상환 능력을 본다.' if inv_top != bank_top else
     f"투자자와 은행 모두 {josa(inv_top, '을를')} 1위로 본다. 성장성과 상환 능력이 함께 좋은 경우다."),
    '은행은 PER·PBR 같은 시장 신호를 보조 지표로 참고하되, 판단은 상환 능력 중심으로 한다.'))

# 5. 스코어카드와 금융 니즈 매핑
grades = ', '.join(f"{n} {sc.loc[n, '종합점수']:.2f}점 {sc.loc[n, '등급']}등급" for n in NAMES)
insights.append((
    '은행의 판단: 기업마다 다른 금융을 제안',
    f'여신 스코어카드(과제용 단순 기준): {grades}.',
    '같은 수출 붐이라도 은행에게 좋은 고객의 조건은 다르다. 성장성·상환 능력·편중 리스크를 함께 봐야 한다.',
    '에이피알 — 환헤지(선물환)·외화예금·해외 결제 자금관리 / 관리: 지역·채널 편중 한도, 운전자본 증가. '
    '코스맥스 — 시설·운전자금 대출, 매출채권 기반 금융, 해외 법인 금융 / 관리: 이익 추정 하향, 차입 증가 속도. '
    '아모레퍼시픽 — 자금 운용(예금·단기 상품), 사업 재편 자금 / 관리: 성장 둔화 시 수익성 재하락 (모두 재무 특징에서 도출한 가설).'))

# 한계 문장은 실제로 쓴 계정에 맞춰 만든다
acc = bank_long.set_index(['종목명', '지표'])['사용계정']
src_of = lambda n: str(acc.get((n, '이자비용_TTM'), '')).split(':')[0]
int_groups = {}
for n in NAMES:
    if src_of(n):
        int_groups.setdefault(src_of(n), []).append(n)
int_note = ('- 이자비용은 회사마다 공시 위치가 달라 ' +
            ', '.join(f"{josa('·'.join(ns), '은는')} {k}" for k, ns in int_groups.items()) +
            '을 썼다. 이자 지급액·금융원가로 대신한 경우 실제 이자비용과 차이가 있다 (사용 계정은 bank_metrics.csv에 기록).'
            if int_groups else '- 전체 재무제표가 없어 이자비용·차입금 등 은행 지표 일부를 계산하지 못했다.')
fin_liab = [n for n in NAMES if '금융부채로 대신' in str(acc.get((n, '총차입금'), ''))]
debt_note = ('- 총차입금에 리스부채는 넣지 않았고, 부채비율의 부채에는 매입채무 같은 영업 부채도 포함된다.' +
             (f" 단, {josa('·'.join(fin_liab), '은는')} 차입금 계정 없이 단기·장기금융부채로만 공시해 그 금액을 썼으며, 리스부채 등이 섞여 있을 수 있다."
              if fin_liab else ''))

md_lines = ['# 금융·비즈니스 인사이트 — 은행의 눈으로 본 3사', f"기준: {d['기준'].iloc[0]}", '',
            '> 투자자는 "주가가 오를까?"를 묻고, 은행은 "빌려준 돈을 돌려받을 수 있나? 이 회사와 어떤 거래를 할 수 있나?"를 묻는다.', '']
for i, (title, fact, meaning, use) in enumerate(insights, 1):
    md_lines += [f'### {i}. {title}', f'- **숫자**: {fact}', f'- **의미**: {meaning}', f'- **활용**: {use}', '']
md_lines += [
    '### 이 분석 도구를 어떻게 쓰나',
    '- **분기마다 재사용**: 실적 발표 후 노트북을 다시 실행하면 은행 지표·스코어카드가 자동 갱신된다 (거래처 모니터링 대시보드).',
    '- **다른 산업으로 확장**: 종목 코드만 바꾸면 차순위 후보였던 K-푸드(예: 삼양식품·농심·오뚜기)에도 같은 틀을 쓸 수 있다. '
    '다만 식품은 밀가루·팜유 같은 원자재를 수입해 매출뿐 아니라 원가 쪽에서도 환율 영향을 받으므로, 수출 환헤지에 더해 원자재 수입 결제·원가 환헤지라는 다른 금융 니즈를 함께 봐야 한다 (검증 전 가설).',
    '- **수출 붐의 확장 기회**: 브랜드가 크면 뒤에 있는 용기·원료·OEM 중소 협력사도 함께 크므로, 중소기업·공급망 금융의 기회가 된다 (업황 해석).',
    '',
    '### 해석의 한계',
    '- 스코어카드는 과제용 단순 기준이며 실제 은행 여신 심사 기준이 아니다.',
    int_note, debt_note,
    '- 금융 니즈 매핑과 협력사 파급은 재무 특징에서 도출한 가설이며, 실제 거래 현황과는 무관하다.',
    '- 점유율은 분석 대상 3사 합계 기준이며 K-뷰티 산업 전체가 아니다. 에이피알의 높은 이익률을 D2C 하나로 단정할 수 없다.',
    '- 해외 비중은 회사마다 산출 기준이 다르고(코스맥스는 해외 법인 매출 합산 참고치), 예상 EPS는 증권사 추정치라 수시로 바뀐다.',
    '- 수업 과제용 분석이며 투자 권유 목적이 아니다.',
]
insight_md = '\n'.join(md_lines)
with open(f'{OUT}/insights.md', 'w', encoding='utf-8') as fp:
    fp.write(insight_md)
print(insight_md)

## STEP 5. Streamlit 결과 화면 (`app/app.py`)
필수 기능: 종목 선택 / 주요 시세 / 주요 재무지표 / 비교 테이블 / 비교 차트 + 밸류체인 위치 표시, 분기·최근 4분기 전환, 이익 점유 탭, 인사이트 탭.
- **Colab:** 아래 셀을 실행하면 앱 주소(링크)가 나옵니다. 런타임을 끄면 앱도 꺼집니다.
  드라이브 폴더에서 바로 띄우면 파일 감시 때문에 멈출 수 있어, 앱과 앱이 읽는 파일(`data/processed`, `output/insights.md`)만 임시 폴더에 복사해 실행합니다.
- **내 컴퓨터:** 프로젝트 폴더에서 `pip install -r requirements.txt` → `streamlit run app/app.py`
- 화면을 캡처해 `output/streamlit_screen.png` 로 저장하세요 (제출물).

In [ ]:
%pip install -q streamlit

In [ ]:
import subprocess, sys, shutil, tempfile, urllib.request

if not os.path.exists('app/app.py'):
    raise FileNotFoundError('app/app.py 가 없습니다 — 드라이브 app 폴더를 확인하세요')

# 드라이브 폴더에서 바로 띄우면 파일 감시 때문에 멈출 수 있음 → 앱과 앱이 읽는 파일만 임시 폴더에 복사해 실행
RUN_DIR = os.path.join(tempfile.gettempdir(), 'streamlit_run')
shutil.rmtree(RUN_DIR, ignore_errors=True)
for d in ['app', PROC, '.streamlit']:
    if os.path.exists(d):
        shutil.copytree(d, os.path.join(RUN_DIR, d))
os.makedirs(os.path.join(RUN_DIR, OUT), exist_ok=True)
shutil.copy(f'{OUT}/insights.md', os.path.join(RUN_DIR, OUT, 'insights.md'))

if 'server' in globals() and server.poll() is None:     # 이 셀을 다시 실행하면 이전 앱을 먼저 끔
    server.terminate()
    server.wait(timeout=10)

LOG_PATH = os.path.join(RUN_DIR, 'streamlit.log')
log = open(LOG_PATH, 'w', encoding='utf-8')
server = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app/app.py',
                           '--server.port=8501', '--server.headless=true', '--server.fileWatcherType=none',
                           '--server.enableCORS=false', '--server.enableXsrfProtection=false'],
                          cwd=RUN_DIR, stdout=log, stderr=subprocess.STDOUT)
healthy = False
for _ in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2) as r:
            healthy = r.status == 200
        if healthy:
            break
    except Exception:
        time.sleep(1)

print('Streamlit 서버:', '정상 실행 (HTTP 200)' if healthy else '실행 실패 → 아래 로그 확인')
if healthy:
    try:
        from google.colab import output
        from google.colab.output import eval_js
        print('앱 주소 (새 탭에서 열기):', eval_js('google.colab.kernel.proxyPort(8501)'))
        output.serve_kernel_port_as_window(8501)             # 링크를 누르면 앱이 새 창으로 열림
    except ImportError:
        print('브라우저에서 열기: http://localhost:8501')
else:
    print(open(LOG_PATH, encoding='utf-8').read()[-2000:])

## 최종 점검 — 결과 파일 목록

In [ ]:
for folder in [RAW, PROC, OUT, 'app']:
    print(f'[{folder}]')
    for f in sorted(os.listdir(folder)):
        print(f'   {f:35s} {os.path.getsize(os.path.join(folder, f)) / 1024:7.1f} KB')